# 1.7 · 索引与执行计划 / Indexes & EXPLAIN

> **课程定位 / Where this fits**
> **Part 1 第 7 课**——从"会写 SQL"升级到"懂数据库**为什么这样跑**"。**性能调优面试题（"这条 SQL 跑得慢，怎么办？"）的标准答题套路就在这里**。
> **Part 1, lesson 7** — from "writes SQL" to "understands what the DB is actually doing". The standard answer template for "this query is slow, fix it?" lives here.

> 📐 **符号约定 / Notation**
> $n$ = 表行数，$k$ = 索引扇出（fan-out，B-tree 每层节点的孩子数），通常 100–1000。
> $n$ = rows in table, $k$ = B-tree fan-out (usually 100-1000).

> 💡 **面试相关 / Interview-relevant**
> - "这条 SQL 跑得慢，怎么 debug？" ★★★★★（标准答：看 `EXPLAIN`）
> - "什么时候索引帮不上忙？" ★★★★★
> - "B-tree 索引是怎么工作的" ★★★★
> - "复合索引列的**顺序**重要吗？" ★★★★（**最左前缀**原则）
> - "OLTP vs OLAP 用什么 DB" ★★★

> ⚠ **数据库引擎的差异 / Engine differences**
> 本节大部分概念以 **PostgreSQL / MySQL（OLTP，B-tree 索引）** 为参照——这是面试的主流问法。
> DuckDB（**OLAP**, 列存储 + 区域映射 zone maps）有不同的优化策略，但 `EXPLAIN` 语法通用。
> Content here references PostgreSQL/MySQL conventions (OLTP, B-tree) — what interviews ask about. DuckDB (OLAP, columnar + zone maps) optimizes differently but EXPLAIN syntax is universal.

---

## 学习目标 / Learning Objectives

学完本节，你应该能：
After this notebook you'll be able to:

1. 用一张图解释 **B-tree** 索引怎么把"全表扫描 $O(n)$" 变成 "$O(\log_k n)$"。
   Explain how a B-tree turns $O(n)$ full scan into $O(\log_k n)$ lookup.
2. 读懂 `EXPLAIN` / `EXPLAIN ANALYZE` 输出的关键字段。
   Read the key fields in `EXPLAIN` / `EXPLAIN ANALYZE`.
3. 列出 **5 种索引帮不上忙**的场景（LIKE '%xx%' / 函数 / OR / 类型转换 / 低选择性）。
   List 5 cases where indexes don't help.
4. 设计**复合索引**的列顺序，并说出**最左前缀**原则。
   Design composite indexes; state the leftmost-prefix rule.
5. 区分三大 **JOIN 策略**：nested loop / hash / merge。
   Distinguish nested-loop / hash / merge joins.
6. 平衡**读 vs 写**的索引成本。
   Weigh read vs write cost of indexes.
7. 给出"我家 SQL 慢"面试题的**标准回答 5 步**。
   Deliver the canonical 5-step answer to "my SQL is slow".

---

## 目录 / Table of Contents

1. [为什么要索引 / Why Indexes](#1)
2. [B-tree 索引内部 / Inside a B-tree](#2)
3. [`EXPLAIN` / `EXPLAIN ANALYZE`](#3)
4. [其他索引类型 / Other Index Types](#4)
5. [何时索引帮不上 ⚠ / When Indexes Don't Help](#5)
6. [复合索引 + 最左前缀 ⭐ / Composite Indexes & Leftmost Prefix](#6)
7. [Covering Index / Index-only Scan](#7)
8. [基数与选择性 / Cardinality & Selectivity](#8)
9. [JOIN 策略：Nested Loop / Hash / Merge](#9)
10. [统计信息与 `ANALYZE` / Statistics](#10)
11. [索引的代价 / Costs of Indexes](#11)
12. [⭐ "SQL 慢怎么办" 标准答 / The Canonical "Slow Query" Answer](#12)
13. [小结 / Summary](#13)


<a id="1"></a>
## 1. 为什么要索引 / Why Indexes

**没有索引** = 数据库每次查询都**全表扫描**：从第 1 行读到最后一行。
**Without an index**: every query is a full-table scan from row 1 to row $n$.

$$T_{\text{scan}} = O(n)$$

**有索引** = 数据库用一棵"导航树"快速定位到目标行。
**With an index**: a "navigation tree" jumps straight to the target rows.

$$T_{\text{lookup}} = O(\log_k n)$$

举例对比：
| $n$ | full scan | B-tree lookup ($k=100$) |
|---|---|---|
| 1 千 | 1 K 次 | ~ 1.5 次（其实就 1 层）|
| 1 万 | 10 K 次 | ~ 2 次 |
| 100 万 | 1 M 次 | ~ 3 次 |
| 1 亿 | 100 M 次 | ~ 4 次 |
| 1 千亿 | 100 G 次（小时级）| ~ 5 次（毫秒）|

**这就是为什么 Google / 银行 / 微信都跑得起来**。
This is why Google / banks / WeChat work at all.


<a id="2"></a>
## 2. B-tree 索引内部 / Inside a B-tree

**B-tree（"Balanced tree"）** 是 OLTP DB 默认索引结构——PostgreSQL、MySQL InnoDB、SQLite、Oracle 都用它。
B-tree is the default index in OLTP DBs.

```
                    [50 | 100]                       ← Root (1 node)
                   /    |     \
               [10|30] [70|90] [120|150]              ← Internal nodes
               /  |  \   ...        ...
             [1] [10..29] [30..49]                    ← Leaves: 实际行指针 / row pointers
              ↓
             page 134, offset 22
```

**关键属性**：
- **平衡** / Balanced：所有叶子等深度 → **查找时间稳定**
- **有序** / Ordered：键按字典序排 → 支持 **范围查询** (`WHERE x BETWEEN ...`)
- **高扇出** / High fan-out：每节点存 100–1000 个键 → 几亿行也只要 4-5 层

### B-tree 支持的查询模式 / Patterns B-tree supports

✅ 等值：`WHERE id = 42`
✅ 范围：`WHERE x BETWEEN 10 AND 100`
✅ 排序：`ORDER BY x` —— 索引已有序，**免排序代价**
✅ 前缀匹配：`WHERE name LIKE 'Apple%'`
❌ 后缀/中间：`WHERE name LIKE '%pple%'` —— 必须全扫
❌ 函数：`WHERE LOWER(email) = 'a@b.com'` —— 索引找的是 `email`，函数变了原值


In [ ]:
import duckdb
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

# 用一个 100,000 行的合成表演示 / Build a 100K-row synthetic table for demos
conn = duckdb.connect()
conn.sql("""
    CREATE TABLE big AS
    SELECT
        i                                              AS id,
        'user_' || (i % 10000)                          AS username,
        CASE i % 5 WHEN 0 THEN 'US' WHEN 1 THEN 'UK'
                   WHEN 2 THEN 'DE' WHEN 3 THEN 'JP'
                   ELSE 'CA' END                       AS country,
        (i * 7) % 1000000                               AS amount,
        DATE '2024-01-01' + INTERVAL (i % 730) DAY      AS event_date
    FROM range(0, 100000) AS t(i);
""")

print(f"duckdb : {duckdb.__version__}")
print(f"big 总行数: {conn.sql('SELECT COUNT(*) FROM big').fetchone()[0]}")
print("\n样本 / sample:")
print(conn.sql("SELECT * FROM big LIMIT 5").df())


<a id="3"></a>
## 3. `EXPLAIN` / `EXPLAIN ANALYZE`

**`EXPLAIN`** 给你"**查询计划**"（plan tree）—— 数据库**打算**怎么跑这条 SQL。
**`EXPLAIN ANALYZE`** = 真跑一遍 + 实测时间 / 行数。

```sql
EXPLAIN SELECT ...;            -- 计划
EXPLAIN ANALYZE SELECT ...;    -- 计划 + 实测
```

不同 DB 输出格式不同，但关键字段类似：
- **算子节点** / Operator: SCAN / JOIN / SORT / HASH AGGREGATE ...
- **预计行数** / Rows estimated
- **实测行数** / Rows actual (只在 ANALYZE 里有)
- **耗时** / Time


In [ ]:
# 看一条简单查询的计划 / Plan for a simple query
print(conn.sql("EXPLAIN SELECT COUNT(*) FROM big WHERE country = 'US';").fetchone()[1])


In [ ]:
# EXPLAIN ANALYZE 真跑一次 / Actually run and time it
print(conn.sql("EXPLAIN ANALYZE SELECT COUNT(*) FROM big WHERE country = 'US';").fetchone()[1])


**怎么读 query plan**（自下而上）/ Reading the plan (bottom-up):
1. **底部**是数据源（表扫描 / 索引查找）/ Bottom = data source
2. **每往上一层**就是新算子（filter、aggregate、join、sort）/ Each layer = an operator
3. **顶部**输出最终结果 / Top = final output

### 关键术语 / Key terms

| 术语 / Term | 含义 |
|---|---|
| **Seq Scan / Full Scan** | 全表扫描 = 慢 |
| **Index Scan** | 走索引查找 = 快 |
| **Index Only Scan** | 直接读索引不读表 = 最快 |
| **Bitmap Scan** | 收集 row IDs 再批量取 |
| **Sort** | 排序，可能 spill 到磁盘 = 慢 |
| **Hash Aggregate** | 哈希分组聚合 |
| **Nested Loop / Hash Join / Merge Join** | 3 种 JOIN 算法（下文）|


<a id="4"></a>
## 4. 其他索引类型 / Other Index Types

B-tree 之外的索引——**面试要知道存在，知道何时用**。
Beyond B-tree — know they exist and when each fits.

| 类型 / Type | 适合 / Best for | 不适合 / Not for |
|---|---|---|
| **B-tree** ⭐ | 等值 + 范围 + 排序 | LIKE '%xx%' |
| **Hash** | 严格等值 (`=` only) | 范围、`ORDER BY` |
| **GIN** (Postgres) | JSONB、数组、全文搜索 | 高基数频繁更新 |
| **BRIN** (Postgres) | 极大表 + 已物理排序（如时序日志）| 随机访问 |
| **Inverted index** | 全文搜索 / 文本 LIKE 加速 | 数值 |
| **R-tree / GiST** | 地理空间 / range queries | 普通列 |
| **Bitmap** | 极低基数列（性别）| 高基数 |

> 💡 99% 的工业场景就用 **B-tree**；JSON / 全文用 **GIN**；时序大表考虑 **BRIN**。
> 99% use B-tree; JSON/text → GIN; huge time-series tables → BRIN.

### DuckDB 的特殊性 / DuckDB specifics

DuckDB 是**列存储 OLAP DB**，**默认不用 B-tree**（只对 PK 用 ART 树）。它靠"**zone maps**"——每个数据块记录 min/max，自动跳过不相关块。但 `EXPLAIN` 语法通用。
DuckDB skips B-trees by default — relies on zone maps. EXPLAIN syntax is shared.


<a id="5"></a>
## 5. ⚠ 何时索引帮不上 / When Indexes Don't Help

**这是面试 ★★★★★ 高频题**。背下来这 5 条。
**Top-5 interview question. Memorize these 5.**

### 5.1 `LIKE '%xx%'`（中间/后缀匹配）

B-tree 按字典序排，找"以 X 开头"很快，找"包含 X"必须**逐行检查**。
B-trees are alphabetically ordered — "starts with X" is fast, "contains X" requires checking every row.

```sql
WHERE name LIKE 'Smith%'    -- ✅ 用索引
WHERE name LIKE '%smith%'   -- ❌ 全扫
WHERE name LIKE '%smith'    -- ❌ 全扫
```

**解决方案** / Fix：**全文索引**（GIN / Inverted）。

### 5.2 函数包住列

```sql
WHERE LOWER(email) = 'a@b.com'                   -- ❌ 索引在 email 上，LOWER 是新值
WHERE DATE(created_at) = '2026-06-01'            -- ❌ 同样
WHERE EXTRACT(YEAR FROM ts) = 2026               -- ❌
```

**解决方案**：
- **表达式索引** (Postgres): `CREATE INDEX ON users (LOWER(email));`
- **改写查询**: `WHERE created_at >= '2026-06-01' AND created_at < '2026-06-02'`

### 5.3 `OR` 条件（混合不同索引列）

```sql
WHERE country = 'US' OR amount > 1000000     -- 可能放弃用任一索引
```

很多优化器会退到全扫。**改成 `UNION`**：
```sql
SELECT * FROM big WHERE country = 'US'
UNION
SELECT * FROM big WHERE amount > 1000000;
```

### 5.4 隐式类型转换 / Implicit type cast

```sql
WHERE phone_number = 13800001111      -- ❌ phone_number 是 VARCHAR
                                       -- DB 把列每行转 number，索引失效
WHERE phone_number = '13800001111'    -- ✅
```

### 5.5 低选择性（命中率太高）

如果 `WHERE gender = 'M'` 匹配 50% 数据，**走索引反而比全扫慢**（来回跳磁盘）。优化器会主动放弃索引。
If a predicate matches >5-10% of rows, the optimizer ditches the index — random index walks beat sequential scan only at low selectivity.

> 💡 **经验阈值**：predicate 选择性 < 5-10% → 索引有用；否则全扫更快。
> Rule of thumb: <5-10% selectivity → index wins.


<a id="6"></a>
## 6. ⭐ 复合索引 + 最左前缀 / Composite Indexes & Leftmost Prefix

**复合索引** `(a, b, c)`：先按 `a` 排，`a` 相同的按 `b` 排，再 `b` 相同的按 `c`。
Composite index `(a, b, c)`: ordered by `a`, then `b` within each `a`, then `c` within each `b`.

```
ABCDEF... (a)
  123456... (b within a)
    XYZ...  (c within b)
```

### 最左前缀原则 / Leftmost prefix rule ⭐

**B-tree 复合索引只能用"从左开始的连续前缀"**。
A composite index `(a, b, c)` only helps queries that use a **leftmost contiguous prefix**.

复合索引 `(a, b, c)` 支持的查询：

| 查询 / Query | 用得上 / Used? |
|---|---|
| `WHERE a = 1` | ✅ |
| `WHERE a = 1 AND b = 2` | ✅ |
| `WHERE a = 1 AND b = 2 AND c = 3` | ✅ |
| `WHERE a = 1 AND c = 3` | ⚠ 只有 `a` 部分用上 |
| `WHERE b = 2` | ❌ **跳过了 a** |
| `WHERE b = 2 AND c = 3` | ❌ |

### 怎么排列顺序 / Picking column order

**规则**：
1. **等值列** 放最前面（先 = 后 ≤）
2. 高**选择性**列优先（基数大）
3. **范围**列放最后（一旦碰到范围，后面的列就不能再筛了）

```sql
-- Good (= 在前)
CREATE INDEX ix_event ON events (user_id, event_date);
SELECT ... WHERE user_id = 1 AND event_date >= '2026-01-01';   ✅ 两列都用

-- Bad (范围在前)
CREATE INDEX ix_event ON events (event_date, user_id);
SELECT ... WHERE user_id = 1 AND event_date >= '2026-01-01';   ⚠ 只 event_date 用上
```

> 💡 **面试金句 / Interview gold**:
> "Always put equality columns first, then high-selectivity, then range — the leftmost-prefix rule means a composite index works only for prefixes."


<a id="7"></a>
## 7. Covering Index / Index-only Scan

**Covering index** = 查询要的**所有列**都在索引里 → **不用回表**。
A "covering index" includes every column the query reads → no table access needed.

```sql
-- 假设 / Suppose
CREATE INDEX ix ON orders (user_id) INCLUDE (amount);
                                    -- ↑ Postgres 11+ 语法

SELECT amount FROM orders WHERE user_id = 42;
-- 只扫索引，不需读 orders 主表 → Index Only Scan
```

**MySQL 等价**：把要 SELECT 的列直接加进复合索引。
MySQL equivalent: just add the SELECT columns into the composite index.

```sql
CREATE INDEX ix ON orders (user_id, amount);    -- amount 也在树里
```

### 为什么这么快 / Why so fast

- 索引通常**远比表小**（只存关键列）→ 缓存命中率高
- 完全不读表 → 节省一次磁盘 I/O
- Postgres / MySQL / SQL Server 都支持

> 💡 **当查询返回少数列时**，covering index 能把 10× 提速变 100×。
> Returns few columns? Covering indexes can push 10× speedups to 100×.


<a id="8"></a>
## 8. 基数与选择性 / Cardinality & Selectivity

| 术语 | 含义 |
|---|---|
| **Cardinality（基数）** | 列里有多少个**唯一值** |
| **Selectivity（选择性）** | 等值匹配命中比例 = 1 / cardinality（均匀分布下）|

- **高基数列**（user_id）→ 选择性高 → **适合建索引**
- **低基数列**（gender, country: 5 个值）→ 选择性低 → **索引帮助有限**

### 工业经验 / Industry rules of thumb

| 列类型 | 是否建索引 |
|---|---|
| 主键 PK | 必须（自动）|
| 外键 FK | 强烈推荐（JOIN 性能）|
| 经常 WHERE 的高基数列 | 推荐 |
| 经常 WHERE 的**低基数**列 | 看情况（< 5%-10% selectivity 才有用）|
| 经常 ORDER BY 的列 | 推荐 |
| 几乎不查的列 | **不要**（白白拖慢写入）|


In [ ]:
# 看 big 表里几列的基数 / Cardinality of various columns
conn.sql("""
    SELECT
        COUNT(DISTINCT id)        AS id_cardinality,         -- 100K (全唯一)
        COUNT(DISTINCT username)  AS username_cardinality,   -- 10K
        COUNT(DISTINCT country)   AS country_cardinality,    -- 5 (低!)
        COUNT(DISTINCT amount)    AS amount_cardinality
    FROM big;
""").df()


**`country` 基数 5**——`WHERE country = 'US'` 命中 20%。**这种列单独建索引不值得**——优化器多半放弃用。
`country` has only 5 distinct values → 20% per match → indexing alone isn't worth it. The optimizer often skips such an index.


<a id="9"></a>
## 9. JOIN 策略 / Join Strategies

数据库实际**执行 JOIN 有 3 种主流算法**，根据数据大小自动选。
DBs auto-pick one of three algorithms based on data size.

| 算法 / Algorithm | 时间复杂度 / Complexity | 适合 / Good when |
|---|---|---|
| **Nested Loop** | $O(n \cdot m)$ | 小表 × 任意，或有索引可避免 $m$ 全扫 |
| **Hash Join** | $O(n + m)$ | **两边都大但能装内存** ⭐ 默认 OLAP 选 |
| **Merge Join** | $O(n \log n + m \log m)$ | 两边都**已经按 JOIN key 排序**（罕见但极快）|

### 9.1 Nested Loop

```
for each row in outer:                 ← n 次
    for each row in inner:             ← m 次
        if (keys match) emit
```
有索引时变成 $O(n \log m)$。**OLTP DB 在小表 JOIN 时偏好它**。
With an index on inner, becomes $O(n \log m)$. OLTP DBs prefer it for small tables.

### 9.2 Hash Join

```
Phase 1: 把 inner 表扔进 hash table     ← O(m)
Phase 2: 对每个 outer 行查 hash         ← O(n)
```
**OLAP DB 的默认选择**（DuckDB / BigQuery / Snowflake）。**前提**：hash table 装得下内存。
The OLAP default — works when the hash side fits in RAM.

### 9.3 Merge Join

```
两边都按 key 排序 → 像归并排序一样并排走
```
最快，**但要先排好序**。如果两边都有 B-tree 索引覆盖 JOIN key，可以"免费"已排序。
Fastest if both inputs are pre-sorted (e.g. both have B-tree indexes on the key).


In [ ]:
# 看一个真实 JOIN 的计划 / Plan for a real JOIN
conn.sql("""
    CREATE TABLE small AS
    SELECT i AS id, 'X' || i AS tag FROM range(0, 100) AS t(i);
""")

print(conn.sql("""
    EXPLAIN
    SELECT b.country, COUNT(*)
    FROM big AS b JOIN small AS s ON b.id % 100 = s.id
    GROUP BY b.country;
""").fetchone()[1])


**注意 plan 里的 HASH_JOIN** —— DuckDB 选了 hash join（OLAP 默认）。
DuckDB picked HASH_JOIN — the OLAP default.


<a id="10"></a>
## 10. 统计信息与 `ANALYZE` / Statistics

优化器**靠统计信息**决定走索引还是全扫、用哪种 JOIN：
The optimizer relies on table statistics to pick plans:

- 表行数
- 每列的**直方图**（数据分布）
- 每列**唯一值数**（基数估算）

### 统计过时 → 计划差 / Stale stats → bad plans

INSERT 了 100M 行**但没更新统计**？优化器以为表只有 1K 行，全用 nested loop——**死给你看**。
After bulk inserts, run `ANALYZE` so the planner knows the real shape:

```sql
ANALYZE big;                      -- 更新所有列统计
ANALYZE big (id, country);        -- 只更新指定列
```

**Postgres** 默认自动 `autovacuum` 周期性 ANALYZE，但**大批量插入后**最好手动跑一次。
Postgres autovacuums periodically, but after bulk loads run it manually.

### 看统计 / Inspect

```sql
-- Postgres
SELECT * FROM pg_stats WHERE tablename = 'big';

-- DuckDB
PRAGMA show_tables;
SUMMARIZE big;                    -- DuckDB 的快捷统计
```


In [ ]:
# DuckDB SUMMARIZE: 一行看全部列的统计 / All-column summary
conn.sql("SUMMARIZE big;").df()


<a id="11"></a>
## 11. 索引的代价 / Costs of Indexes

**索引不是免费午餐**。每个索引：
Indexes aren't free. Every index:

| 代价 / Cost | 说明 |
|---|---|
| **写放大 / Write amplification** | INSERT/UPDATE/DELETE 都要更新**每个**索引 → 越多索引写越慢 |
| **存储 / Storage** | 每个索引 = 表大小的 10-50%（取决于列宽度）|
| **优化器决策时间** | 索引越多，优化器选起来越慢（通常忽略不计）|
| **锁竞争** | 更新索引页 = 短暂排他锁 |

### 工业经验法则 / Industry rules

1. **每张表的索引数 ≤ 5–7** 个（OLTP）—— 多了写性能塌方
2. **每次 INSERT 都触发每个索引** → 重写表数据 + N × 索引数据
3. 大表上每加一个索引都要**冷静评估**：能 cover 的查询多吗？

### 何时**删**索引 / When to drop

```sql
-- Postgres: 看哪些索引很少被用 / Identify rarely-used indexes
SELECT indexrelname, idx_scan
FROM pg_stat_user_indexes
WHERE idx_scan < 10
ORDER BY idx_scan;
```

`idx_scan = 0` 几个月 → 安全删除。
Zero scans for months → safe to drop.


<a id="12"></a>
## 12. ⭐ "SQL 慢怎么办" 标准答 / The Canonical "Slow Query" Answer

**面试 ★★★★★** —— 几乎必问。下面 5 步把它一气讲完，**面试官会满意**。
**Top-5 interview question.** Walk through these 5 steps and you'll impress.

### Step 1 · 看 EXPLAIN ANALYZE / Read the plan

```sql
EXPLAIN ANALYZE SELECT ...;
```

- 是 **Seq Scan** 还是 **Index Scan**？
- 哪一步耗时最久？（**self time** 大的那一行）
- 估计行数 vs 实际行数差很多？→ **统计过时**

### Step 2 · 找瓶颈算子 / Find the bottleneck

- **Seq Scan on big** → 没索引或没用上 → **加索引**
- **Sort** 耗时高 → 看能否用索引"免排序"
- **Nested Loop**, $n × m$ 都大 → 看是否应该 Hash Join（更新统计 / 提示）
- **Hash Aggregate** 内存爆 → 看是否能预聚合 / 加 PARTITION

### Step 3 · 检查索引使用 / Check index usage

- `WHERE` 里有**函数包列**？→ 改 expression index 或重写
- `LIKE '%x%'`？→ 全文索引
- 复合索引列顺序对吗？→ 见第 6 节最左前缀

### Step 4 · 重写 SQL / Rewrite if needed

- `OR` → `UNION`
- `IN (subquery)` → 看是否能改 `EXISTS`
- 巨大 `GROUP BY` → 看能否先 filter 再 group

### Step 5 · 数据层优化（最后手段）/ Data-layer fixes

- **分区** / Partitioning: 按时间分表，扫描扫一小部分
- **物化视图** / Materialized view: 预聚合 + 定时刷新
- **Cache**: 把热门查询结果放 Redis

### 💡 一句话答 / The one-line answer

> "**先看 EXPLAIN ANALYZE 定位瓶颈，再判断是缺索引、走错索引、还是统计过时；不行就重写 SQL；再不行就分区或物化。**"
> "EXPLAIN ANALYZE first to find the bottleneck, then check missing/wrong indexes or stale stats; rewrite SQL if needed; partition or materialize as last resort."


<a id="13"></a>
## 13. 小结 / Summary

### 概念地图 / Concept map

```
查询慢
  │
  ├── 读 EXPLAIN ANALYZE
  │     ├── 是 Seq Scan 吗？
  │     ├── Sort / GroupAgg 在哪里？
  │     └── 估计行数 ≠ 实际？ → ANALYZE 更新统计
  │
  ├── 索引
  │     ├── B-tree (默认) → 等值 + 范围 + 排序 + 前缀 LIKE
  │     ├── 复合索引 → 最左前缀原则 ⭐
  │     ├── Covering index → Index-only scan
  │     └── 5 大失效情况：LIKE '%x%' / 函数 / OR / 隐式转换 / 低选择性
  │
  ├── JOIN 策略
  │     ├── Nested loop → 小表 / 有索引
  │     ├── Hash join → OLAP 默认 ⭐
  │     └── Merge join → 已排序，最快
  │
  └── 数据层
        ├── 分区
        ├── 物化视图
        └── Cache (Redis ...)
```

### 💡 必背速查 / Must-remember

| 问题 | 答 |
|---|---|
| **怎么 debug 慢 SQL?** | EXPLAIN ANALYZE → 找 Seq Scan / Sort 瓶颈 |
| **B-tree 怎么变 O(log n)？** | 平衡 + 高扇出 100-1000 |
| **复合索引列序怎么排？** | 等值在前、高基数在前、范围最后（最左前缀）|
| **5 个索引帮不上场景？** | `LIKE '%x%'` / 函数 / OR / 隐式转换 / 低选择性 |
| **OLAP / OLTP JOIN 默认？** | OLAP → Hash；OLTP → Nested loop (小表 / 有索引) |
| **索引代价？** | 每次写要更新每个索引 → 索引太多写性能塌 |
| **谁需要 ANALYZE？** | 大量写入之后 → 统计过时会选错 plan |

### 💡 面试速查 / Interview must-knows

1. **5 步标准答**：EXPLAIN → 找瓶颈 → 检查索引 → 重写 SQL → 数据层
2. **最左前缀** —— 必答（复合索引列顺序题）
3. **`WHERE LOWER(x) = ...`** 是经典坑 → 表达式索引或改写
4. **Hash Join 是 OLAP 默认** —— 现代分析引擎都用
5. **B-tree 复杂度 $O(\log_k n)$** —— $k = 100$ 扇出 → 1 亿行 4 层

### 下一节预告 / Next up

**Part 1.8 · Python + SQL 集成** —— SQLAlchemy、pandas.read_sql、psycopg2、DuckDB、连接池、防 SQL 注入。把 SQL 写进 Python 项目的工程化。
**Part 1.8 · Python ↔ SQL** — SQLAlchemy, pandas.read_sql, psycopg2, DuckDB, connection pooling, SQL injection prevention.
